In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

plt.style.use("ggplot")

df = pd.read_csv("/kaggle/input/datasets/yashdevladdha/uber-ride-analytics-dashboard/ncr_ride_bookings.csv", na_values=["null"])

df["Booking ID"] = df["Booking ID"].str.replace('"', "", regex=False)
df["Customer ID"] = df["Customer ID"].str.replace('"', "", regex=False)
df["Date"] = pd.to_datetime(df["Date"])
df["Time"] = pd.to_datetime(df["Time"], format="%H:%M:%S")

df_modelo = df[df["Booking Status"] == "Completed"].copy().reset_index(drop=True)
df_modelo["hora"] = df_modelo["Time"].dt.hour
df_modelo["dia_semana"] = df_modelo["Date"].dt.dayofweek
df_modelo["mes"] = df_modelo["Date"].dt.month
df_modelo["final_semana"] = df_modelo["dia_semana"].isin([5, 6]).astype(int)

df_modelo["hora_rush"] = (
    df_modelo["hora"].between(7, 10) |
    df_modelo["hora"].between(17, 20)
).astype(int)

def limites_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

val_inf, val_sup = limites_iqr(df_modelo["Booking Value"])
dist_inf, dist_sup = limites_iqr(df_modelo["Ride Distance"])

mask = (
    df_modelo["Booking Value"].between(val_inf, val_sup) &
    df_modelo["Ride Distance"].between(dist_inf, dist_sup)
)

df_limpo = df_modelo[mask].copy().reset_index(drop=True)

features_numericas = ["Ride Distance", "Avg VTAT", "Avg CTAT",
                      "hora", "dia_semana", "mes",
                      "final_semana", "hora_rush"]

features_categoricas = ["Vehicle Type", "Payment Method"]

target = "Booking Value"

df_final = df_limpo[features_numericas + features_categoricas + [target]].dropna().copy()
X = df_final.drop(columns=[target])
y = df_final[target]

pipeline_num = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

pipeline_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", pipeline_num, features_numericas),
    ("cat", pipeline_cat, features_categoricas)
], remainder="drop")

print(f"Setup completo. {len(df_final):,} corridas prontas para o modelo.")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print("=== DIVISÃO DOS DADOS ===")
print(f" Treino : {len(X_train):,} corridas (70%)")
print(f" Teste  : {len(X_test):,} corridas (30%)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Treino (70%)", "Teste (30%)"], [len(X_train), len(X_test)],
       color=["#2196F3", "#FF9800"], edgecolor="white")

for i, v in enumerate([len(X_train), len(X_test)]):
    ax.text(i, v + 200, f"{v:,}", ha="center")

ax.set_title("Divisão Treino / Teste")
ax.set_ylabel("Número de Corridas")
plt.tight_layout()
plt.show()

In [ ]:
modelo_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("modelo", LinearRegression())
])

modelo_lr.fit(X_train, y_train)

y_pred_lr = modelo_lr.predict(X_test)

print("Regressão Linear treinada!")
print(f" Previsões geradas: {len(y_pred_lr):,} valores (uma para cada corrida no teste)")
print(" Exemplo: primeiras 5 previsões (R$):", y_pred_lr[:5].round(2))

In [ ]:
modelo_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("modelo", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

print("Treinando Random Forest (100 árvores)... aguarde...")
modelo_rf.fit(X_train, y_train)

y_pred_rf = modelo_rf.predict(X_test)

print("Random Forest treinado!")
print(f" Previsões geradas: {len(y_pred_rf):,} valores")
print(f" Exemplo: primeiras 5 previsões (R$): {y_pred_rf[:5].round(2)}")

In [ ]:
resumo_pred = pd.DataFrame({
    "valor_real": y_test.values[:10],
    "pred_RegLinear": y_pred_lr[:10].round(2),
    "pred_RandomForest": y_pred_rf[:10].round(2),
})

print("=== AMOSTRA: VALOR REAL VS PREVISÕES (10 primeiras corridas de teste) ===")
print(resumo_pred.to_string(index=False))
print("\n(Abaixo: várias avaliações no próprio arquivo.)")

In [ ]:
def mape_reg(y_true, y_pred):
    mask = np.asarray(y_true) != 0
    return np.mean(np.abs((np.asarray(y_true)[mask] - np.asarray(y_pred)[mask]) / np.asarray(y_true)[mask])) * 100


# cálculo das métricas
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

r2_lr = r2_score(y_test, y_pred_lr)
r2_rf = r2_score(y_test, y_pred_rf)

mape_lr = mape_reg(y_test.values, y_pred_lr)
mape_rf = mape_reg(y_test.values, y_pred_rf)


print("=== AVALIAÇÃO 1: MÉTRICAS MÚLTIPLAS ===")
print(f"{'Métrica':<10} {'Reg. Linear':>14} {'Random Forest':>14}")
print("-" * 40)

print(f"{'MAE (R$)':<10} {mae_lr:>14.1f} {mae_rf:>14.1f}")
print(f"{'RMSE (R$)':<10} {rmse_lr:>14.1f} {rmse_rf:>14.1f}")
print(f"{'R²':<10} {r2_lr:>14.2%} {r2_rf:>14.2%}")
print(f"{'MAPE (%)':<10} {mape_lr:>14.1f} {mape_rf:>14.1f}")

In [ ]:
y_baseline = np.full_like(y_test, y_train.mean())

mae_base = mean_absolute_error(y_test, y_baseline)
r2_base = r2_score(y_test, y_baseline)

print("=== AVALIAÇÃO 2: COMPARAÇÃO COM BASELINE ===")
print(f"Baseline (sempre média):  MAE = R${mae_base:,.0f}, R² = {r2_base:.2%}")
print(f"Regressão Linear:        MAE = R${mae_lr:,.0f}, R² = {r2_lr:.2%}")
print(f"Random Forest:           MAE = R${mae_rf:,.0f}, R² = {r2_rf:.2%}")

print("-> Os modelos superam o baseline? Se sim, as features ajudam.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, titulo, cor, mae_val, r2_val in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ["Regressão Linear", "Random Forest"],
    ["#2196F3", "#FF5722"],
    [mae_lr, mae_rf],
    [r2_lr, r2_rf]
):
    ax.scatter(y_test, y_pred, alpha=0.15, color=cor, s=10)

    lim_min = 50
    lim_max = max(y_test.max(), np.max(y_pred))

    ax.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        "k--",
        label="Previsão perfeita"
    )

    ax.set_title(f"{titulo} — MAE = R${mae_val:.0f}, R² = {r2_val:.2%}")
    ax.set_xlabel("Valor Real (R$)")
    ax.set_ylabel("Valor Previsto (R$)")
    ax.legend()
    ax.set_xlim(lim_min, lim_max + 20)
    ax.set_ylim(0, lim_max + 50)

plt.suptitle("Avaliação: Previsto vs Real")
plt.tight_layout()
plt.show()

In [ ]:
residuos_lr = y_test.values - y_pred_lr
residuos_rf = y_test.values - y_pred_rf

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, res, titulo, cor in zip(
    axes,
    [residuos_lr, residuos_rf],
    ["Regressão Linear", "Random Forest"],
    ["#2196F3", "#FF5722"]
):

    ax.hist(res, bins=50, color=cor, edgecolor="white", alpha=0.7)

    ax.axvline(0, color="black", linestyle="--", linewidth=2)

    ax.axvline(
        np.mean(res),
        color="red",
        linestyle=":",
        label=f"Média = R${np.mean(res):.0f}"
    )

    ax.set_xlabel("Resíduo (R$)")
    ax.set_ylabel("Frequência")
    ax.set_title(titulo)
    ax.legend()

plt.suptitle("Avaliação: Distribuição dos erros")
plt.tight_layout()
plt.show()

In [ ]:
print("=== AVALIAÇÃO 5: VALIDAÇÃO CRUZADA (5 folds, R²) ===")

cv_lr = cross_val_score(modelo_lr, X, y, cv=5, scoring="r2", n_jobs=-1)
cv_rf = cross_val_score(modelo_rf, X, y, cv=5, scoring="r2", n_jobs=-1)

for i, (a, b) in enumerate(zip(cv_lr, cv_rf), 1):
    print(f" Fold {i}: Reg. Linear = {a:.4f}, Random Forest = {b:.4f}")

print(f" Média: Reg. Linear = {cv_lr.mean():.4f} ± {cv_lr.std():.4f}")
print(f"        Random Forest = {cv_rf.mean():.4f} ± {cv_rf.std():.4f}")

In [ ]:
ohe = modelo_rf.named_steps["preprocessor"].named_transformers_["cat"].named_steps["encoder"]

nomes_cat = ohe.get_feature_names_out(features_categoricas).tolist()
nomes_feat = features_numericas + nomes_cat

imp = modelo_rf.named_steps["modelo"].feature_importances_

feat_imp = pd.Series(imp, index=nomes_feat).sort_values(ascending=False)
top10 = feat_imp.head(10).sort_values()

print("=== AVALIAÇÃO 6: TOP 10 FEATURES MAIS IMPORTANTES (Random Forest) ===")
for nome, valor in feat_imp.head(10).items():
    print(f"{nome}: {valor*100:.1f}%")

plt.figure(figsize=(8, 5))
top10.plot(kind="barh", color="steelblue", edgecolor="white")
plt.title("Avaliação: Importância das features para prever preço")
plt.xlabel("Importância relativa")
plt.tight_layout()
plt.show()

In [ ]:
# juntar valores reais, previsões e tipo de veículo
df_eq = pd.DataFrame({
    "valor_real": y_test.values,
    "valor_pred": y_pred_rf,
    "Vehicle Type": df_modelo.loc[y_test.index, "Vehicle Type"]
})

# erro absoluto
df_eq["erro_abs"] = np.abs(df_eq["valor_real"] - df_eq["valor_pred"])

# calcular MAE por tipo de veículo
mae_por_tipo = (
    df_eq.groupby("Vehicle Type")["erro_abs"]
    .mean()
    .sort_values(ascending=False)
)

print("=== AVALIAÇÃO: ERRO MÉDIO POR TIPO DE VEÍCULO ===")
print(mae_por_tipo.round(1))

# gráfico
plt.figure(figsize=(8,5))
mae_por_tipo.sort_values().plot(
    kind="barh",
    color="steelblue",
    edgecolor="white"
)

plt.title("Avaliação: Erro médio por tipo de veículo (equidade)")
plt.xlabel("MAE (R$)")
plt.ylabel("Vehicle Type")
plt.tight_layout()
plt.show()

In [ ]:
dt_aval = pd.DataFrame({
    "valor_real": y_test.values,
    "valor_pred": y_pred_rf
})

dt_aval["erro_abs"] = np.abs(dt_aval["valor_real"] - dt_aval["valor_pred"])

target = "valor_real"

In [ ]:
dt_aval["faixa"] = pd.cut(
    dt_aval[target],
    bins=[0, 200, 400, 600, 2000],
    labels=["até 200", "200-400", "400-600", "600+"]
)

erro_por_faixa = (
    dt_aval.groupby("faixa", observed=True)
    .agg(
        erro_medio=("erro_abs", "mean"),
        n=("erro_abs", "count")
    )
)

print("=== AVALIAÇÃO 8: ERRO MÉDIO POR FAIXA DE VALOR DA CORRIDA ===")
print(erro_por_faixa.round(1).to_string())

In [ ]:
print("Dia 4 (Linear01) concluído. Modelos treinados e 8 avaliações aplicadas.")